In [1]:
!pip install earthpy gdal --quiet
!pip install xgboost --quiet

In [2]:
from osgeo import gdal, gdal_array
import os
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import random
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from skimage import io
import joblib
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import math
from PIL import Image
from sklearn.ensemble import RandomForestClassifier
import cv2
import skimage
from sklearn.svm import SVC
import xgboost as xgb
import torch
import torch.nn.functional as F
import xgboost as xgb
from xgboost import XGBClassifier
import gc

mask_path_train="/kaggle/input/resized-clean-splitted-augmented-cloud-data/cleaned-data/train/masks"
data_path_train ="/kaggle/input/resized-clean-splitted-augmented-cloud-data/cleaned-data/train/images"
mask_path_test="/kaggle/input/resized-clean-splitted-augmented-cloud-data/cleaned-data/test/masks"
data_path_test ="/kaggle/input/resized-clean-splitted-augmented-cloud-data/cleaned-data/test/images"
mask_path_val="/kaggle/input/resized-clean-splitted-augmented-cloud-data/cleaned-data/val/masks"
data_path_val ="/kaggle/input/resized-clean-splitted-augmented-cloud-data/cleaned-data/val/images"


2025-04-30 14:30:41.012662: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746023441.034391     114 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746023441.041048     114 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
train_data= os.listdir(data_path_train)
test_data= os.listdir(data_path_test)
val_data= os.listdir(data_path_val)

print(len(train_data))
print(len(test_data))
print(len(val_data))

3302
1039
1034


In [4]:
def display_image(data_path,mask_path,file_name,data_only):
    file_path=os.path.join(data_path, file_name)
    img_ds= gdal.Open(file_path, gdal.GA_ReadOnly)
    num_bands=img_ds.RasterCount
    if( not data_only):
        fig, axs = plt.subplots(1, 5, figsize=(10, 5))
    else:
        fig, axs = plt.subplots(1, 4, figsize=(10, 5))
    
    for i in range(1,num_bands+1):
      band = img_ds.GetRasterBand(i)
      img = band.ReadAsArray()
      axs[i-1].imshow(img, cmap='gray')
      axs[i-1].set_title('Original Image')
    if(not data_only):
        file_path=os.path.join(mask_path, file_name)
        img_ds= gdal.Open(file_path, gdal.GA_ReadOnly)
        num_bands=img_ds.RasterCount
        band = img_ds.GetRasterBand(1)
        img = band.ReadAsArray()
        axs[4].imshow(img, cmap='gray', vmin=0, vmax=1)        
        axs[4].set_title('Mask')
    plt.show()

In [5]:
# Define Sobel kernels
sobel_kernel_x = torch.tensor([[ -1, 0, 1],
                               [ -2, 0, 2],
                               [ -1, 0, 1]], dtype=torch.float32, device='cuda').unsqueeze(0).unsqueeze(0)
sobel_kernel_y = torch.tensor([[ -1, -2, -1],
                               [ 0, 0, 0],
                               [ 1, 2, 1]], dtype=torch.float32, device='cuda').unsqueeze(0).unsqueeze(0)


In [6]:
def extract_features(image):
    # 1. Convert image to PyTorch tensor, float32, move to GPU
    img_tensor = torch.from_numpy(image).float().unsqueeze(0).unsqueeze(0).cuda()  # shape [1, 1, H, W]

    # 2. Pad the image
    padded = F.pad(img_tensor, (2, 2, 2, 2), mode='replicate')  # pad 2 pixels on each side
   
    # 3. Extract 5x5 windows using unfold
    windows = padded.unfold(2, 5, 1).unfold(3, 5, 1)  # shape: [1,1,H,W,5,5]
   
    W,H = image.shape
    windows = windows.contiguous().view(1,1,W,H,25)  # flatten 5x5 into 25 features

    # 4. Separate center pixel and neighbors
    center_pixel = windows[...,12]  # index 12 is center of 5x5
    neighbors = torch.cat([windows[...,:12], windows[...,13:]], dim=-1)  # remove center

    # 5. Mean and std
    mean_val = windows.mean(dim=-1)
    std_val = windows.std(dim=-1)

    # 6. Sobel gradient
    
    gx = F.conv2d(img_tensor, sobel_kernel_x, padding=1)  # shape [1,1,H,W]
    gy = F.conv2d(img_tensor, sobel_kernel_y, padding=1)

    # 7. Reshape gx and gy to match the other tensors' dimensions (i.e., adding a channel dimension)
    gx = gx.unsqueeze(-1)  # shape: [1, 1, H, W, 1]
    gy = gy.unsqueeze(-1)  # shape: [1, 1, H, W, 1]

    # 8. Stack all features (now gx and gy have consistent dimensions)
    features = torch.cat([
        center_pixel.unsqueeze(-1),
        neighbors,
        mean_val.unsqueeze(-1),
        std_val.unsqueeze(-1),
        gx,
        gy
    ], dim=-1)  # shape: [1, 1, H, W, feature_dim]

    # 9. Flatten to (H*W, feature_dim)
    features = features.view(-1, features.shape[-1])  # shape [H*W, feature_dim]

    # 10. Move back to CPU and numpy
    return features.cpu().numpy()


In [7]:
def dice_coefficient(y_true, y_pred, smooth=1e-6):
    y_pred = np.ascontiguousarray(y_pred)
    y_true = np.ascontiguousarray(y_true)

    intersection = (y_pred * y_true).sum(axis=(1, 2))
    dice = (2. * intersection + smooth) / (y_pred.sum(axis=(1, 2)) + y_true.sum(axis=(1, 2)) + smooth)
    return dice.mean()

In [8]:
H=256
W=256
batch_size = 256 
params = {
    'tree_method': 'hist',      # New recommended method
    'device': 'cuda',
    'objective': 'binary:logistic',
    'eval_metric': 'logloss'
}


In [9]:
def load_data_in_batches(X,data_path,mask_path, batch_size):
    num_samples = len(X)
    for start in range(0, num_samples, batch_size):
        end = min(start + batch_size, num_samples)
        X_train=[]
        y_train=[]
        imgs=[]
        for i in range(start,end):
            # img
            file_path=os.path.join(data_path, X[i])
            img_ds= gdal.Open(file_path, gdal.GA_ReadOnly)
            band = img_ds.GetRasterBand(1)
            img = band.ReadAsArray()
            imgs.append(img)
            X_train.append (extract_features(img))
            # mask
            file_path=os.path.join(mask_path, X[i])
            mask_ds= gdal.Open(file_path, gdal.GA_ReadOnly)
            band = mask_ds.GetRasterBand(1)
            mask = band.ReadAsArray()
            y_train.append( mask.reshape(-1))
        X_train=np.array(X_train)
        y_train=np.array(y_train)
        num_features=X_train.shape[-1]
        y_train=y_train.reshape(-1)
        X_train=X_train.reshape(-1, num_features)
        yield imgs,X_train,y_train


In [10]:
def train(train_data,data_path, mask_path, batch_size,eta,dep, num_rounds=10):
    print("start evaluating with parameter=",params)

    model = None
    params["eta"]=eta
    params["max_depth"]=dep
    # --- Training loop over batches ---
    for i, (_, X_batch, y_batch) in tqdm(enumerate(load_data_in_batches(train_data, data_path, mask_path, batch_size)),desc="Training loop"):
        dtrain_batch = xgb.DMatrix(X_batch, label=y_batch)

        if model is None:
            model = xgb.train(params, dtrain_batch, num_boost_round=num_rounds)
        else:
            model.update(dtrain_batch, i)  # You can also keep a separate round counter

        del X_batch, y_batch, dtrain_batch
        gc.collect()

    return model


In [11]:
def evaluate(model,val_data,data_path,mask_path, batch_size,debub):
    print("start evaluating....")
    val_dice=0
    num_batches=0
    for imgs,X_batch, y_batch in tqdm(load_data_in_batches(val_data,data_path,mask_path, batch_size),desc="validation loop"):
        dval_batch_data = xgb.DMatrix(X_batch)
        # model prediction
        y_pred = model.predict(dval_batch_data)
        # Threshold the predictions
        y_pred_binary = (y_pred > 0.5).astype(np.uint8)
        y_pred_mask = y_pred_binary.reshape(batch_size,H, W)
        y_true_mask = y_batch.reshape(batch_size,H, W)
        val_dice+= dice_coefficient(y_true_mask, y_pred_mask)
        num_batches+=1
        if debug:
            # visualize the first image in each batch
            fig, axs = plt.subplots(2, 3, figsize=(10, 5))
            axs[0,0].imshow(imgs[0], cmap='gray')
            axs[0,0].set_title('Original Image')
            axs[0,1].imshow(y_true_mask[0], cmap='gray',vmin=0,vmax=1)
            axs[0,1].set_title('ground truth')
            axs[0,2].imshow(y_pred_mask[0], cmap='gray',vmin=0,vmax=1)
            axs[0,2].set_title('predicted mask')
            plt.show()
        # Free memory manually
        del X_batch
        del y_batch
        del dval_batch_data
        gc.collect()
    val_dice/=num_batches
    return val_dice


In [ ]:
max_depth={4,6,8}
etas={0.01,0.05,0.001}

best_model=None
best_dice=0
best_params=None

for dep in max_depth:
    for eta in etas:
        model,params=train(train_data,data_path_train,mask_path_train, batch_size,eta,dep)
        train_dice=evaluate(model,train_data,data_path_train,mask_path_train, batch_size)
        print("traing dice score = ",train_dice)
        val_dice=evaluate(model,val_data,data_path_val,mask_path_val, batch_size)
        print("validation dice score = ",val_dice)
        if val_dice>best_dice:
            best_dice=val_dice
            best_model=model
            best_params=params

start evaluating with parameter= {'tree_method': 'hist', 'device': 'cuda', 'objective': 'binary:logistic', 'eval_metric': 'logloss'}


Training loop: 2it [00:32, 16.42s/it]

In [ ]:
print("the best dice score = "best_dice)
print("the best params = "best_params)

In [ ]:
# testing on testset
test_dice=evaluate(model,test_data,data_path_test,mask_path_test, batch_size)
print("test dice score = ",test_dice)

In [ ]:
# save the model
model.save_model('xgboost_model.model')
import pickle
with open('xgboost_model.pkl', 'wb') as f:
    pickle.dump(model, f)